# Numerical Issues in Transformers

| Difficulty | ██████░░░░ 6/10 |
| :--- | :--- |
| Prerequisites | 06.06 |
| Time | ~85 min |
| Colab GPU | Needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/06_Numerical_Methods/07_numerical_issues_in_transformers.ipynb)

---

## 🎯 Learning Objective

Diagnose attention-score overflow and residual-stream magnitude growth, compare pre-norm vs. post-norm placement for training stability, analyze error propagation through deep residual vs. sequential networks, and understand RMSNorm's numerical advantages over LayerNorm.

---

## 📐 Theory

This capstone-style notebook for Module 06 applies every earlier idea — IEEE 754 (`06.01`),
stable softmax (`06.02`), and mixed precision (`06.06`) — to the specific architecture where
they matter most in practice: the transformer.

### Attention score overflow

Self-attention computes $\text{softmax}(QK^\top/\sqrt{d_k})V$ (recall `01.10`). The scores
$QK^\top/\sqrt{d_k}$ are unbounded — nothing in the formula caps their magnitude — and if $Q$,
$K$ carry large activations (common before the network's normalization layers have had a
chance to constrain them, especially early in training or in poorly-initialized layers), the
scores can reach magnitudes in the hundreds or thousands. Even when those scores are perfectly
representable in a given float format (recall `06.01`), $\exp(\text{score})$ inside softmax
overflows far sooner: $e^{89}$ already exceeds float32's maximum representable value, and
float16 overflows even earlier. This is exactly `06.02`'s stable-softmax problem, reappearing
in a specific, load-bearing spot inside every transformer layer — which is why production
attention implementations always use the numerically stable (subtract-the-max) softmax formula,
never the naive one.

### Pre-norm vs. post-norm placement

The original Transformer places LayerNorm *after* each sublayer, applied to the sum of the
residual and the sublayer's output — **post-norm**:
$\mathbf{h} \leftarrow \text{LayerNorm}(\mathbf{h} + \text{Sublayer}(\mathbf{h}))$. Later
architectures (GPT-2 onward) commonly use **pre-norm** instead: normalize *before* the sublayer,
and add its output to the *un-normalized* residual stream:
$\mathbf{h} \leftarrow \mathbf{h} + \text{Sublayer}(\text{LayerNorm}(\mathbf{h}))$.

These look like a minor reshuffling, but they have a real, measurable effect connecting
directly to `02.10`'s vanishing/exploding gradient analysis. In post-norm, every layer's output
gets renormalized to a controlled scale, but the identity path (recall `02.10`'s residual
connection discussion) then runs *through* that normalization rather than around it — so the
clean "local gradient exactly $1$" property of a pure residual connection is partially
disrupted by the normalization's own Jacobian. In pre-norm, the residual stream itself is never
directly normalized — each sublayer's contribution is added to a running, unconstrained sum,
which keeps a cleaner identity gradient path but lets the residual stream's *magnitude* grow
steadily with depth (each new layer just adds more to an already-large running total).

### Residual stream magnitude growth

Because pre-norm's residual path is never itself normalized, tracking the residual stream's
norm layer-by-layer typically shows it growing with depth — visible directly in the
Implementation section below. This isn't necessarily a problem on its own (LayerNorm inside
each sublayer keeps the *sublayer's own* computation well-scaled regardless of the running
residual's size), but it does mean later layers see inputs of a different scale than earlier
ones, a real practical consideration when choosing initialization and learning rates for very
deep transformers.

### Why practitioners care about this specific tradeoff

Empirically, pre-norm transformers tend to train more stably at initialization and support
learning-rate warmup with less careful tuning (part of why they became the more common choice
for large modern LLMs), while post-norm can require more careful warmup schedules (recall
`04.06`) to avoid early instability, but has sometimes shown better final performance when
training succeeds. Neither is a universal "correct" answer — this is a real architectural
tradeoff practitioners still navigate, not settled math.

---

### Error propagation through depth

When a network is $L$ layers deep and each layer introduces a rounding error $\delta_l$
(from finite-precision arithmetic — recall `06.01`), the total error at the output depends
critically on *how* those per-layer errors compose:

**Residual (additive) error accumulation.** With residual connections,
$h_{l+1} = h_l + f_l(h_l) + \delta_l$. Errors accumulate *additively*:
total error $\approx \sum_{l=1}^L \delta_l$. If the per-layer errors are roughly independent
and zero-mean (a reasonable model when rounding errors don't have a systematic bias), the
central limit theorem gives $|\text{total error}| \sim O(\sqrt{L})$ — growing only as the
square root of depth. This is one of the underappreciated reasons residual networks can be
made very deep without catastrophic numerical error accumulation, beyond the gradient-flow
argument from `02.10`.

**Sequential (multiplicative) error accumulation.** Without residual connections,
$h_{l+1} = f_l(h_l) + \delta_l$. If each $f_l$ has Lipschitz constant $K_l > 1$ (i.e., it
can amplify differences), errors from early layers are *multiplied* through subsequent layers:
$|\text{total error}| \leq \sum_{l=1}^L |\delta_l| \prod_{j=l+1}^L K_j$. For constant
$K > 1$, this grows as $O(K^L)$ — exponential in depth. This is the numerical-stability lens
on the same exploding-gradient phenomenon `02.10` analyzed from the optimization perspective.

**Mixed-precision backprop error.** In a typical mixed-precision training setup (`06.06`),
the forward pass runs in float16 (or bfloat16), the backward pass also runs in reduced
precision, but the weight *update* happens in float32 (master weights). The rounding error
introduced during the forward pass propagates backward through the chain rule, with each
layer's Jacobian potentially amplifying or dampening it. The practical consequence:
**bfloat16** forward + backward with a float32 optimizer works out of the box for most
architectures, because bfloat16 shares float32's full exponent range ($\pm 3.4 \times 10^{38}$)
— gradients rarely overflow or underflow even without intervention. **float16**, with its
narrow exponent range (max $\approx 65504$), causes frequent gradient underflow in the backward
pass, which is precisely why `06.06`'s dynamic loss scaling exists: it artificially inflates
the loss before backpropagation to keep small gradients within float16's representable range,
then divides the gradient by the same factor before the float32 weight update.

### RMSNorm: cheaper normalization, different numerics

**RMSNorm** (Zhang & Sennrich, 2019), used in LLaMA, Gemma, and Mistral, is a simpler
alternative to LayerNorm that drops the mean-centering step entirely:

$$\text{RMSNorm}(x)_i = \frac{x_i}{\text{RMS}(x)} \cdot \gamma_i, \qquad
\text{RMS}(x) = \sqrt{\frac{1}{d}\sum_{j=1}^d x_j^2}$$

Compare this to LayerNorm ($\text{LN}(x)_i = (x_i - \mu)/\sigma \cdot \gamma_i + \beta_i$):
RMSNorm has no mean subtraction ($\mu$) and no bias ($\beta_i$).

**Numerical advantages over LayerNorm:**

1. **No mean subtraction → no catastrophic cancellation risk.** LayerNorm computes
   $x_i - \mu$, and when $\mu$ is large relative to the spread of $x$'s entries (which can
   happen when the residual stream grows with depth, as shown in the pre-norm analysis above),
   this subtraction is exactly the catastrophic-cancellation pattern from `06.02`: two
   nearly-equal numbers are subtracted, losing most of their significant bits. RMSNorm
   divides by a positive scalar without first centering, completely sidestepping this risk.

2. **One fewer reduction operation → less inter-thread synchronization on GPUs.** LayerNorm
   requires two reductions over the hidden dimension (mean, then variance); RMSNorm requires
   only one (mean of squares). On modern GPUs, reductions across a large hidden dimension
   are the bottleneck — so this is a real wall-clock speedup, not just a FLOP count savings.

3. **Empirically matches LayerNorm quality.** LLaMA (Touvron et al., 2023) demonstrated at
   scale that replacing LayerNorm with RMSNorm throughout a large transformer causes no
   measurable quality degradation while improving throughput. This is why every major open
   LLM family since (Gemma, Mistral, Qwen) has adopted RMSNorm as the default.

---

In [ ]:
# Setup
import numpy as np
import torch
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

Feeding deliberately large-magnitude Q/K vectors into an attention computation shows naive
softmax collapsing into NaN once $\exp(\text{score})$ overflows float16's range, while the
numerically stable softmax formula stays a valid probability distribution throughout.

In [ ]:
torch.manual_seed(0)
seq_len, d_k = 8, 64

def naive_softmax(scores):
    e = torch.exp(scores)
    return e / e.sum(dim=-1, keepdim=True)

scale_factors = np.linspace(1, 35, 25)
naive_has_nan, stable_has_nan = [], []
for s in scale_factors:
    Q = (torch.randn(seq_len, d_k) * s).half()
    K = (torch.randn(seq_len, d_k) * s).half()
    scores = (Q @ K.T) / np.sqrt(d_k)
    with torch.no_grad():
        naive_probs = naive_softmax(scores)
        stable_probs = torch.softmax(scores, dim=-1)
    naive_has_nan.append(torch.isnan(naive_probs).any().item())
    stable_has_nan.append(torch.isnan(stable_probs).any().item())

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(scale_factors, naive_has_nan, 'o-', color="#C44E52", label="naive softmax (has NaN)")
ax.plot(scale_factors, stable_has_nan, 's-', color="#4C72B0", label="stable softmax (has NaN)")
ax.set_xlabel("Q/K magnitude scale factor"); ax.set_ylabel("any NaN in attention weights? (1=yes)")
ax.set_yticks([0, 1]); ax.set_yticklabels(["No", "Yes"])
ax.set_title("Naive attention softmax breaks down in float16\nwell before the stable version does")
ax.legend()
plt.show()

first_naive_nan = scale_factors[np.argmax(naive_has_nan)] if any(naive_has_nan) else None
print(f"Naive softmax first shows NaN around Q/K scale ~ {first_naive_nan:.1f}")
print(f"Stable softmax never showed NaN across the entire tested range: {not any(stable_has_nan)}")

## 🐍 Implementation from Scratch

We build a minimal attention mechanism with both a naive and a numerically stable softmax,
confirm the exact failure mode with real numbers, then build both pre-norm and post-norm
transformer blocks and track how residual-stream magnitude and gradients behave through depth.

In [ ]:
def attention(Q, K, V, use_stable_softmax=True):
    d_k = Q.shape[-1]
    scores = Q @ K.transpose(-2, -1) / np.sqrt(d_k)
    if use_stable_softmax:
        probs = torch.softmax(scores, dim=-1)
    else:
        e = torch.exp(scores)
        probs = e / e.sum(dim=-1, keepdim=True)
    return probs @ V, scores

torch.manual_seed(0)
seq_len, d_k = 8, 64
scale = 30  # large enough to push scores past exp()'s float16 overflow point
Q = (torch.randn(seq_len, d_k) * scale).half()
K = (torch.randn(seq_len, d_k) * scale).half()
V = torch.randn(seq_len, d_k).half()

out_stable, scores = attention(Q, K, V, use_stable_softmax=True)
out_naive, _ = attention(Q, K, V, use_stable_softmax=False)
print(f"Score range: [{scores.min().item():.1f}, {scores.max().item():.1f}]  (well within float16's representable range)")
print(f"Naive attention output has NaN:  {torch.isnan(out_naive).any().item()}")
print(f"Stable attention output has NaN: {torch.isnan(out_stable).any().item()}")

# --- Pre-norm vs post-norm: track residual-stream magnitude AND gradient through depth ---
def make_block(d_model):
    ln1, ln2 = torch.nn.LayerNorm(d_model), torch.nn.LayerNorm(d_model)
    attn_proj = torch.nn.Linear(d_model, d_model)
    ff = torch.nn.Sequential(torch.nn.Linear(d_model, d_model * 2), torch.nn.ReLU(), torch.nn.Linear(d_model * 2, d_model))
    return ln1, ln2, attn_proj, ff

def run_stack(n_layers, d_model, pre_norm, seed=0, track_grad=False):
    torch.manual_seed(seed)
    dtype = torch.float64 if track_grad else torch.float32
    x = torch.randn(4, d_model, dtype=dtype, requires_grad=track_grad)
    blocks = [make_block(d_model) for _ in range(n_layers)]
    if track_grad:
        for b in blocks:
            for m in b:
                m.double()
    h, magnitudes = x, [x.norm(dim=-1).mean().item()]
    for ln1, ln2, attn_proj, ff in blocks:
        if pre_norm:
            h = h + attn_proj(ln1(h))
            h = h + ff(ln2(h))
        else:
            h = ln1(h + attn_proj(h))
            h = ln2(h + ff(h))
        magnitudes.append(h.norm(dim=-1).mean().item())
    if track_grad:
        probe = torch.randn(d_model, dtype=dtype)  # generic linear functional; avoids the
        (h * probe).sum().backward()                # trivial-zero artifact of summing a mean-zero LN output
        return magnitudes, x.grad.norm().item()
    return magnitudes, None

d_model = 32
for n_layers in [5, 10, 20, 40]:
    mags_pre, grad_pre = run_stack(n_layers, d_model, pre_norm=True, track_grad=True)
    mags_post, grad_post = run_stack(n_layers, d_model, pre_norm=False, track_grad=True)
    print(f"\nn_layers={n_layers}: pre-norm  final magnitude={mags_pre[-1]:.2f}, input grad norm={grad_pre:.3f}")
    print(f"              post-norm final magnitude={mags_post[-1]:.2f}, input grad norm={grad_post:.3f}")

In [ ]:
# --- RMSNorm from scratch, compared to LayerNorm on edge cases ---

class RMSNorm(torch.nn.Module):
    """Root Mean Square Layer Normalization (Zhang & Sennrich, 2019)."""
    def __init__(self, d, eps=1e-8):
        super().__init__()
        self.gamma = torch.nn.Parameter(torch.ones(d))
        self.eps = eps

    def forward(self, x):
        rms = torch.sqrt(torch.mean(x ** 2, dim=-1, keepdim=True) + self.eps)
        return (x / rms) * self.gamma

# Compare RMSNorm vs LayerNorm on edge-case inputs
d = 256
rmsnorm = RMSNorm(d)
layernorm = torch.nn.LayerNorm(d)

print("=== RMSNorm vs LayerNorm on edge-case inputs ===")
print(f"{'Input description':<35} {'LN has NaN/Inf':>15} {'RMS has NaN/Inf':>16}")
print("-" * 70)

test_cases = [
    ("Normal random",          torch.randn(4, d)),
    ("Large constant + noise",  torch.full((4, d), 1e4) + torch.randn(4, d) * 0.01),
    ("Very large offset (fp16)", (torch.full((4, d), 500.0) + torch.randn(4, d) * 0.1).half()),
    ("Tiny values near zero",   torch.randn(4, d) * 1e-6),
    ("Mixed: huge + tiny",      torch.cat([torch.full((4, d//2), 1e3), torch.full((4, d//2), 1e-5)], dim=-1)),
]

for name, x in test_cases:
    if x.dtype == torch.float16:
        # Run both norms in float16 to expose cancellation
        ln_out = torch.nn.LayerNorm(d, device=x.device, dtype=torch.float16)(x)
        rms_out = RMSNorm(d).half()(x)
    else:
        ln_out = layernorm(x)
        rms_out = rmsnorm(x)
    ln_bad = torch.isnan(ln_out).any().item() or torch.isinf(ln_out).any().item()
    rms_bad = torch.isnan(rms_out).any().item() or torch.isinf(rms_out).any().item()
    print(f"{name:<35} {str(ln_bad):>15} {str(rms_bad):>16}")

# Demonstrate the cancellation mechanism explicitly
print("\n=== Why LayerNorm risks cancellation ===")
x_offset = torch.full((1, d), 1000.0) + torch.randn(1, d) * 0.01
mu = x_offset.mean(dim=-1, keepdim=True)
centered = x_offset - mu  # catastrophic cancellation: 1000.xx - 1000.yy
rms_val = torch.sqrt(torch.mean(x_offset ** 2, dim=-1, keepdim=True))
print(f"Input mean: {mu.item():.4f}, Input std: {x_offset.std().item():.6f}")
print(f"After mean subtraction — max |centered|: {centered.abs().max().item():.6f}")
print(f"  → Significant bits lost: ~{int(np.log2(mu.item() / (centered.abs().max().item() + 1e-10)))}"
      f" (subtracting {mu.item():.0f} from values near {mu.item():.0f})")
print(f"RMS value (no subtraction needed): {rms_val.item():.4f} — no cancellation risk")


# --- Error propagation: residual vs non-residual through 100 layers ---
print("\n" + "=" * 70)
print("=== Error propagation: residual vs non-residual (100 layers) ===")
print("=" * 70)

def simulate_error_propagation(n_layers, d_model, use_residual, dtype_fwd, seed=42):
    """Run the same linear layers in high precision (float64) and in dtype_fwd,
    tracking the per-layer error between them."""
    torch.manual_seed(seed)
    # Build shared weights (in float64 for ground truth)
    weights = [torch.randn(d_model, d_model, dtype=torch.float64) * 0.5 / np.sqrt(d_model)
               for _ in range(n_layers)]

    x_hi = torch.randn(8, d_model, dtype=torch.float64)
    x_lo = x_hi.to(dtype_fwd)

    errors = []
    h_hi, h_lo = x_hi, x_lo
    for l in range(n_layers):
        W_hi = weights[l]
        W_lo = W_hi.to(dtype_fwd)

        if use_residual:
            h_hi = h_hi + torch.tanh(h_hi @ W_hi)   # residual + nonlinearity
            h_lo = h_lo + torch.tanh(h_lo @ W_lo)
        else:
            h_hi = torch.tanh(h_hi @ W_hi)           # sequential (no residual)
            h_lo = torch.tanh(h_lo @ W_lo)

        # Relative error (compare against float64 ground truth)
        diff = (h_lo.double() - h_hi).norm() / (h_hi.norm() + 1e-30)
        errors.append(diff.item())

    return errors

n_layers_sim, d_sim = 100, 128

err_residual_fp16 = simulate_error_propagation(n_layers_sim, d_sim, use_residual=True,
                                                dtype_fwd=torch.float16)
err_sequential_fp16 = simulate_error_propagation(n_layers_sim, d_sim, use_residual=False,
                                                  dtype_fwd=torch.float16)
err_residual_bf16 = simulate_error_propagation(n_layers_sim, d_sim, use_residual=True,
                                                dtype_fwd=torch.bfloat16)
err_sequential_bf16 = simulate_error_propagation(n_layers_sim, d_sim, use_residual=False,
                                                  dtype_fwd=torch.bfloat16)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=False)

# Left panel: residual vs sequential
ax = axes[0]
ax.semilogy(err_residual_fp16, color="#4C72B0", lw=2, label="Residual (float16)")
ax.semilogy(err_sequential_fp16, color="#C44E52", lw=2, label="Sequential (float16)")
ax.set_xlabel("Layer depth"); ax.set_ylabel("Relative error vs float64 (log scale)")
ax.set_title("Residual connections keep error\ngrowth sub-linear (float16)")
ax.legend(fontsize=9); ax.set_xlim(0, n_layers_sim)

# Right panel: float16 vs bfloat16 (residual only)
ax = axes[1]
ax.semilogy(err_residual_fp16, color="#4C72B0", lw=2, label="Residual — float16")
ax.semilogy(err_residual_bf16, color="#55A868", lw=2, label="Residual — bfloat16")
ax.semilogy(err_sequential_bf16, color="#DD8452", lw=2, ls="--", label="Sequential — bfloat16")
ax.set_xlabel("Layer depth"); ax.set_ylabel("Relative error vs float64 (log scale)")
ax.set_title("bfloat16 has higher per-layer error (fewer mantissa bits)\nbut wider range prevents overflow")
ax.legend(fontsize=9); ax.set_xlim(0, n_layers_sim)

plt.tight_layout()
plt.show()

print(f"\nAfter {n_layers_sim} layers (float16):")
print(f"  Residual   relative error: {err_residual_fp16[-1]:.2e}")
print(f"  Sequential relative error: {err_sequential_fp16[-1]:.2e}")
print(f"  Ratio (sequential / residual): {err_sequential_fp16[-1] / (err_residual_fp16[-1] + 1e-30):.1f}x")
print(f"\nAfter {n_layers_sim} layers (bfloat16):")
print(f"  Residual   relative error: {err_residual_bf16[-1]:.2e}")
print(f"  Sequential relative error: {err_sequential_bf16[-1]:.2e}")
print(f"  bfloat16 residual error is larger than float16's (fewer mantissa bits)")
print(f"  but bfloat16 never overflows — this is the fundamental tradeoff from 06.01")

## 🤖 Why This Matters for AI

Every production transformer implementation — GPT-family models, BERT, LLaMA, the attention
kernel inside PyTorch's `scaled_dot_product_attention` — uses a numerically stable softmax
internally for exactly the reason demonstrated above, and virtually every modern large model
uses pre-norm placement specifically because it tolerates the aggressive learning rates and
minimal warmup tuning that pre-training at scale requires, even though it means accepting
residual-stream growth as a tradeoff. Below, we visualize the magnitude and gradient trends
from the Implementation section together, then connect the residual-stream growth finding back
to `06.01`'s bfloat16-vs-float16 discussion: a growing residual stream is one more concrete
reason transformer training prefers bfloat16's wide exponent range over float16's narrow one.

In [ ]:
n_layers_plot = 40
mags_pre, _ = run_stack(n_layers_plot, d_model, pre_norm=True)
mags_post, _ = run_stack(n_layers_plot, d_model, pre_norm=False)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(mags_pre, color="#4C72B0", lw=2, label="pre-norm (residual stream grows)")
ax.plot(mags_post, color="#C44E52", lw=2, label="post-norm (magnitude stays controlled)")
ax.set_xlabel("layer depth"); ax.set_ylabel("mean residual-stream magnitude")
ax.set_title("Pre-norm's residual stream accumulates with depth;\npost-norm's stays roughly constant")
ax.legend()
plt.show()

print(f"After {n_layers_plot} layers: pre-norm magnitude={mags_pre[-1]:.1f}, post-norm magnitude={mags_post[-1]:.1f}")

# Does that growing magnitude survive representation in float16 vs bfloat16?
final_stream_pre = torch.full((1,), mags_pre[-1], dtype=torch.float32)
print(f"\nFinal pre-norm residual-stream magnitude, cast to float16:  {final_stream_pre.half().item()}")
print(f"Final pre-norm residual-stream magnitude, cast to bfloat16: {final_stream_pre.bfloat16().item()}")
print("At THIS magnitude both formats are fine -- but recall from 06.01 that float16 overflows")
print("entirely above 65504, while bfloat16 shares float32's full exponent range. A deep enough")
print("pre-norm stack (or one with less-controlled initialization) can push past that boundary")
print("in float16 well before bfloat16 would ever notice -- one more concrete reason bfloat16")
print("is the standard choice for training, not just an arbitrary alternative to float16.")

## 🏋️ Exercises

Eight exercises across six levels, in order: mechanical computation → conceptual understanding → derivation →
implementation → application → open-ended challenge. Solutions/outlines are in the collapsed
`Solution` blocks below each exercise — try before revealing.

### 🔢 COMPUTE
1. For $Q, K$ with $d_k=64$ and every entry equal to a constant $c$, compute every entry of
   $QK^\top/\sqrt{d_k}$ symbolically in terms of $c$. Then find the smallest integer $c$ that
   makes $\exp$ of that score overflow float16 (max representable value $\approx 65504$, so
   overflow happens once the exponent argument exceeds $\ln(65504)\approx 11.09$).

<details>
<summary>💡 Solution</summary>

Each entry of $QK^\top$ is a dot product of two length-64 all-$c$ vectors:
$64\cdot c^2$. Dividing by $\sqrt{d_k}=8$ gives a score of $8c^2$ for every entry. At $c=1$,
the score is $8 < 11.09$ (safe). At $c=2$, the score is $32 > 11.09$ — already well past the
overflow threshold. So $c=2$ is the smallest integer at which this construction overflows
float16, even though $Q$ and $K$'s own entries ($\pm2$) are utterly unremarkable numbers.

</details>

### 💭 UNDERSTAND
2. In the Implementation cell, `d_model` changes from 32 to 128 while keeping `n_layers=20`
   fixed. Does a wider model change the *qualitative* pattern (pre-norm's residual-stream
   magnitude grows with depth, post-norm's stays controlled), or just the specific numbers?

<details>
<summary>💡 Solution</summary>

Only the specific numbers change, not the qualitative pattern. At $d_{model}=32$: pre-norm's
final magnitude is $16.60$ (input-gradient norm $27.7$) vs. post-norm's $5.66$ (gradient
$9.2$). At $d_{model}=128$: pre-norm's final magnitude is $31.98$ (gradient $68.0$) vs.
post-norm's $11.31$ (gradient $21.4$) — both magnitudes roughly double, but pre-norm still grows
relative to post-norm in both cases, and pre-norm's gradient norm is still consistently about
$3\times$ post-norm's at both widths. The mechanism driving the growth (an unnormalized running
sum in pre-norm vs. a per-layer renormalization in post-norm) doesn't depend on $d_{model}$ at
all, so the qualitative behavior is architecture-driven, not width-driven — only the absolute
scale of numbers shifts.

</details>

3. **(New)** RMSNorm computes $x_i / \text{RMS}(x)$ while LayerNorm computes
   $(x_i - \mu)/\sigma$. When the residual stream has grown to a large magnitude (say,
   $\mu \approx 1000$ with spread $\approx 0.01$), explain precisely which arithmetic
   operation in LayerNorm triggers catastrophic cancellation (`06.02`), and why RMSNorm's
   formula sidesteps this entirely.

<details>
<summary>💡 Solution</summary>

LayerNorm's centering step computes $x_i - \mu$ for each element. When $\mu \approx 1000$ and
individual $x_i \approx 1000 \pm 0.01$, this subtraction is $1000.01 - 1000.00 = 0.01$ — two
numbers agreeing in their first ~5 significant digits are subtracted, and the result retains
only the last ~1–2 significant digits. In float32 (which has ~7 decimal digits of precision),
this loses about 5 digits of accuracy; in float16 (~3.3 digits), essentially *all* precision is
lost. This is exactly the catastrophic cancellation pattern from `06.02`.

RMSNorm computes $\text{RMS}(x) = \sqrt{\text{mean}(x^2)}$ and then divides $x_i / \text{RMS}(x)$.
There is no subtraction of nearly-equal quantities anywhere in this formula — squaring,
averaging, and taking a square root are all *magnitude-preserving* operations that don't create
cancellation. The division $x_i / \text{RMS}(x)$ is a scaling by a positive number, which is
numerically benign. This is why RMSNorm is strictly safer than LayerNorm from a
numerical-stability perspective, even though both produce outputs of similar scale.

</details>

### ✏️ DERIVE
4. `02.10` established that a pure residual connection $h \leftarrow h + f(h)$ has local
   gradient exactly $I + \partial f/\partial h$ — the identity term guarantees a gradient path
   that never vanishes regardless of $f$'s own Jacobian. Derive the analogous local-gradient
   expression for one **post-norm** block,
   $h \leftarrow \text{LayerNorm}(h + f(h))$, and explain precisely which part of the
   Theory section's "partially disrupted identity gradient" claim this derivation makes precise.

<details>
<summary>💡 Solution outline</summary>

By the chain rule, the local gradient of $\text{LayerNorm}(u)$ with $u = h+f(h)$ is
$\frac{\partial \text{LayerNorm}}{\partial u}\Big|_{u} \cdot \frac{\partial u}{\partial h} = J_{LN}(u)\cdot(I + \partial f/\partial h)$,
where $J_{LN}(u)$ is LayerNorm's own Jacobian at $u$ (a projection-like matrix that removes the
mean and rescales by $1/\sigma$, recall LayerNorm's definition). Unlike pre-norm's clean
$I + \partial f/\partial h$, post-norm's gradient is that *same* expression left-multiplied by
$J_{LN}(u)$ — so the "always contains an exact identity term" property is preserved structurally
(the $I$ is still inside the parenthesized factor), but the *overall* local gradient is no
longer literally $I$ plus a correction; it's $J_{LN}(u)$ applied to that sum, and $J_{LN}(u)$ is
generally neither the identity nor even full-rank in every direction (LayerNorm's mean-centering
step alone kills one degree of freedom). This is precisely what "partially disrupted" means: the
residual path's contribution doesn't vanish, but it no longer passes through completely
unchanged either.

</details>

### 🐍 IMPLEMENT
5. Extend `run_stack` to also record the residual stream's magnitude **variance across the 4
   batch examples** (not just their mean) at every layer, for both pre-norm and post-norm.
   Compare how that cross-example variance evolves with depth in each architecture.

<details>
<summary>✅ How to know you're right</summary>

Your per-layer variance values should be computed the same way `magnitudes` already computes
per-layer means — from `h.norm(dim=-1)` across the batch dimension, just using `.var()` instead
of `.mean()` — and should start from the same (nonzero, since the 4 random examples aren't
identical) variance at layer 0 for both architectures, since they share the identical input
`x`. There's no single mandated qualitative outcome here (that's genuinely worth investigating
rather than assuming), but your numbers should be internally consistent: pre-norm's per-layer
mean magnitude should match the notebook's existing `mags_pre` values exactly, confirming you
extended the function correctly rather than accidentally changing what it already computed.

</details>

6. **(New)** Implement a mixed-precision error tracker: build a 3-layer MLP (Linear → ReLU →
   Linear → ReLU → Linear), run the same input through it in float16 and float64, and record
   the relative error $\|h^{\text{fp16}}_l - h^{\text{fp64}}_l\| / \|h^{\text{fp64}}_l\|$
   after each layer. Then repeat in bfloat16. Plot both error curves and verify that bfloat16
   has *larger* per-layer error (fewer mantissa bits) but *never* produces NaN/Inf (wider
   exponent range).

<details>
<summary>✅ How to know you're right</summary>

Your per-layer relative errors should be small (roughly $10^{-3}$ to $10^{-2}$ for float16,
$10^{-2}$ to $10^{-1}$ for bfloat16) and should grow modestly with depth (3 layers isn't
enough for dramatic accumulation, but the trend should be visible). The key verification:
bfloat16's errors should be *consistently larger* than float16's at every layer (because
bfloat16 has only 8 mantissa bits vs. float16's 11), but both should produce valid (non-NaN,
non-Inf) outputs for reasonable inputs. If you scale the input up (try multiplying by 100),
float16 may produce Inf/NaN while bfloat16 stays finite — demonstrating the exponent-range
advantage.

</details>

### 🤖 APPLY
7. `06.04`'s conjugate-gradient notebook showed that a Gram matrix $K+\sigma^2 I$'s
   conditioning improves as $\sigma^2$ grows. Here, a growing pre-norm residual stream means
   later transformer layers process inputs at a systematically larger scale than earlier layers
   see. Explain the connection: why would a *fixed* weight initialization scheme (e.g. every
   linear layer initialized the same way regardless of depth) become progressively less
   appropriate for later layers in a very deep pre-norm transformer, and what would a
   depth-aware initialization need to account for?

<details>
<summary>✅ What you should observe</summary>

A weight initialization scheme (like Xavier/He initialization) is derived assuming a particular
expected input scale, tuned so the layer's output variance matches its input variance. If the
residual stream's actual magnitude grows substantially with depth (as pre-norm's does — recall
the Visual Intuition plot growing from an initial magnitude to $16$–$32\times$ larger by
40 layers), a fixed initialization tuned for the *early*-layer input scale is implicitly wrong
for late layers, whose LayerNorm will renormalize the input to its sublayer correctly (LayerNorm
always maps to zero mean, unit variance internally) but whose *added contribution* back into the
residual stream is still calibrated for a smaller expected running total. A depth-aware scheme
(as real deep-transformer literature explores) would need to account for the expected residual
magnitude at each depth, not just the sublayer's own local input/output scale.

</details>

### 🚀 CHALLENGE
8. Modify the pre-norm block to scale each sublayer's contribution by $1/\sqrt{n\_layers}$
   before adding it to the residual stream (a simple depth-dependent scaling used in some deep
   transformer variants), and re-measure the final residual-stream magnitude at `n_layers=40`.
   Does this successfully control the growth while still resembling a pre-norm architecture, and
   how does it compare to switching to post-norm entirely?

<details>
<summary>🔍 What a strong answer covers</summary>

Numerically: plain pre-norm at `n_layers=40` reaches a final magnitude of $\approx22.44$
(starting from $\approx5.88$); adding the $1/\sqrt{40}$ scaling brings it down to $\approx6.76$
— close to both the *initial* magnitude and to post-norm's own final magnitude of $\approx5.66$
at the same depth. A strong answer recognizes this as a genuine middle path: the scaling factor
shrinks each sublayer's contribution just enough that the running sum's growth is tamed (each
added term counts for less as depth increases, controlling the geometric-ish accumulation),
while the architecture is still structurally pre-norm — normalization still happens *before* each
sublayer, and the identity gradient path through the residual connection (this notebook's
Exercise 3 derivation) is completely untouched, unlike post-norm's genuinely modified gradient
path. The tradeoff: the scale factor is a global hyperparameter that must be tuned (or
theoretically derived) rather than emerging automatically the way post-norm's per-layer
renormalization does, and using a single $1/\sqrt{n\_layers}$ for the whole network assumes
every layer needs the same correction, which may not hold if some layers systematically
contribute more than others.

</details>

---

## 📚 Further Reading
- Xiong et al., ["On Layer Normalization in the Transformer Architecture"](https://arxiv.org/abs/2002.04745) (pre-norm vs. post-norm analysis)
- Vaswani et al., ["Attention Is All You Need"](https://arxiv.org/abs/1706.03762)
- Dao et al., ["FlashAttention: Fast and Memory-Efficient Exact Attention with IO-Awareness"](https://arxiv.org/abs/2205.14135) (numerically stable, fused attention in practice)
- Zhang & Sennrich, ["Root Mean Square Layer Normalization"](https://arxiv.org/abs/1910.07467) (RMSNorm)
- Touvron et al., ["LLaMA: Open and Efficient Foundation Language Models"](https://arxiv.org/abs/2302.13971) (RMSNorm adoption at scale)

---

*Next notebook: [Quantization for Inference and Training](08_quantization_for_inference_and_training.ipynb)*